# 모델 로드와 추론

> 사전학습된 모델을 로드하고, 다양한 샘플링 전략으로 텍스트를 생성하는 전체 과정

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 1. 정밀도(Precision)에 따른 메모리 차이

In [2]:
# === FP32 vs FP16: 같은 모델, 다른 메모리 ===
# FP32로 로드 (기본값)
model_fp32 = AutoModelForCausalLM.from_pretrained('gpt2')
mem_fp32 = sum(p.numel() * p.element_size() for p in model_fp32.parameters())

# FP16으로 로드 (torch_dtype 지정)
model_fp16 = AutoModelForCausalLM.from_pretrained('gpt2', torch_dtype=torch.float16)
mem_fp16 = sum(p.numel() * p.element_size() for p in model_fp16.parameters())

print(f"GPT-2 메모리 비교:")
print(f"  FP32: {mem_fp32/1024**2:.1f}MB (파라미터당 4바이트)")
print(f"  FP16: {mem_fp16/1024**2:.1f}MB (파라미터당 2바이트)")
print(f"  절감률: {(1 - mem_fp16/mem_fp32)*100:.0f}%")
print(f"\n-> 실제 7B 모델에서는 28GB vs 14GB 차이")

# 메모리 정리
del model_fp32, model_fp16
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 964.98it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1186.77it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 메모리 비교:
  FP32: 474.7MB (파라미터당 4바이트)
  FP16: 237.4MB (파라미터당 2바이트)
  절감률: 50%

-> 실제 7B 모델에서는 28GB vs 14GB 차이


In [4]:
# === device_map: 모델을 어디에 로드할것인가 ===
print("device_map 옵션:")
print("=" * 50)
options = [
    ('"auto"', 'GPU/CPU 자동 분배 (가장 편리)'),
    ('"cpu"', 'CPU에만 로드 (GPU 없을 때)'),
    ('"cuda:0"', '특정 GPU에 로드'),
    ('지정 안 함', 'CPU에 로드 (GPT-2 같은 작은 모델)'),
]

for opt, desc in options:
    print(f"  {opt:<15} -> {desc}")

# 현재 환경 확인
print(f"\n현재 환경:")
print(f"  CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB")

device_map 옵션:
  "auto"          -> GPU/CPU 자동 분배 (가장 편리)
  "cpu"           -> CPU에만 로드 (GPU 없을 때)
  "cuda:0"        -> 특정 GPU에 로드
  지정 안 함          -> CPU에 로드 (GPT-2 같은 작은 모델)

현재 환경:
  CUDA 사용 가능: True
  GPU: NVIDIA GeForce RTX 4070 Ti
  VRAM: 12.0GB


---
## 2. 텍스트 생성: generate() 샘플링 전략

In [13]:
# === 모델 + 토크나이저 로드 ===
tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def generate_text(prompt, **kwargs):
    """텍스트 생성 헬퍼 함수"""
    inputs = tokenizer(prompt, return_tensors='pt')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=kwargs.pop('max_new_tokens', 50),
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    # 입력 부분 제외하고 생성된 부분만 디코딩
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated

print("준비 완료")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1130.10it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


준비 완료


In [6]:
# === Greedy Decoding: 매번 가장 높은 확률 선택 ===
# do_sample=False (기본값) -> 항상 같은 결과
prompt = "The future of AI is"

print("Greedy Decoding (do_sample=False):")
print("-> 매번 실행해도 동일한 결과\n")
for i in range(3):
    result = generate_text(prompt, do_sample=False)
    print(f"  [{i+1}] {result[:80]}...")

Greedy Decoding (do_sample=False):
-> 매번 실행해도 동일한 결과

  [1]  uncertain. The future of AI is uncertain.

The future of AI is uncertain. The f...
  [2]  uncertain. The future of AI is uncertain.

The future of AI is uncertain. The f...
  [3]  uncertain. The future of AI is uncertain.

The future of AI is uncertain. The f...


In [7]:
# === Temperature 비교: 낮으면 확정적, 높으면 랜덤 ===
# Phase 1에서 softmax(logits / temperature)로 배운 개념
prompt = "Once upon a time"

temperatures = [0.3, 0.7, 1.0, 1.5]
print(f"프롬프트: '{prompt}'\n")

for temp in temperatures:
    result = generate_text(prompt, do_sample=True, temperature=temp, top_p=1.0)
    print(f"  temp={temp}: {result[:80]}...")

print("\n-> 낮은 temp: 안전하지만 지루할 수 있음")
print("-> 높은 temp: 창의적이지만 횡설수설 가능")

프롬프트: 'Once upon a time'

  temp=0.3: , the world was a dark place.

The world was a dark place.

The world was a dark...
  temp=0.7: , there were two options: Either I'd never have been able to find my father, or ...
  temp=1.0: , I was one of those who wished to be known as 'the good guy.' I was also the la...
  temp=1.5: , I remember feeling that we should do, I am ashamed and frustrated when she did...

-> 낮은 temp: 안전하지만 지루할 수 있음
-> 높은 temp: 창의적이지만 횡설수설 가능


In [8]:
# === top_k vs top_p 비교 ===
prompt = "The best way to learn programming is"

print(f"프롬프트: '{prompt}'\n")

# top_k: 상위 k개 토큰만 후보
print("top_k=10 (상위 10개만):")
result = generate_text(prompt, do_sample=True, temperature=0.7, top_k=10)
print(f"  {result[:80]}...\n")

# top_p: 누적 확률 p까지만 후보 (= nucleus sampling)
print("top_p=0.9 (누적 90%까지):")
result = generate_text(prompt, do_sample=True, temperature=0.7, top_p=0.9)
print(f"  {result[:80]}...\n")

# top_k=50, top_p=0.9 조합 (실전 권장)
print("top_k=50 + top_p=0.9 (실전 권장):")
result = generate_text(prompt, do_sample=True, temperature=0.7, top_k=50, top_p=0.9)
print(f"  {result[:80]}...")

프롬프트: 'The best way to learn programming is'

top_k=10 (상위 10개만):
   to learn about programming, not just how to learn.

The best way to learn progr...

top_p=0.9 (누적 90%까지):
   to know how to do things on your own.

I'll show you how to do everything on yo...

top_k=50 + top_p=0.9 (실전 권장):
   to learn how to use the language.

Learn programming

If you are in the busines...


In [9]:
# === repetition_penalty: 반복 억제 ===
# GPT-2는 반복이 심한 편이라 이 설정이 중요
prompt = "I think that"

print(f"프롬프트: '{prompt}'\n")

# 반복 억제 없이
print("repetition_penalty=1.0 (억제 없음):")
result = generate_text(prompt, do_sample=True, temperature=0.7, repetition_penalty=1.0)
print(f"  {result[:100]}...\n")

# 반복 억제 적용
print("repetition_penalty=1.3 (억제 적용):")
result = generate_text(prompt, do_sample=True, temperature=0.7, repetition_penalty=1.3)
print(f"  {result[:100]}...")

print("\n-> 1.0=억제 없음, 1.1~1.3=적당한 억제, 2.0+=강한 억제")

프롬프트: 'I think that'

repetition_penalty=1.0 (억제 없음):
   in the end we are a lot safer to live by," says Gertrude.

The study by the Yale School of Public H...

repetition_penalty=1.3 (억제 적용):
   there was a great deal of effort put into it, but I'm not sure which one. When you're playing with ...

-> 1.0=억제 없음, 1.1~1.3=적당한 억제, 2.0+=강한 억제


---
## 3. Beam Search

In [14]:
# === Beam Search: 여러 후보를 동시에 탐색 ===
# Greedy는 매 스텝 최선을 선택 -> 전역 최적이 아닐 수 있음
# Beam Search는 여러 경로를 동시에 유지하며 최선 선택
prompt = "The capital of France is"

print(f"프롬프트: '{prompt}'\n")

# Greedy
print("Greedy (do_sample=False):")
result = generate_text(prompt, do_sample=False, max_new_tokens=30)
print(f"  {result[:80]}\n")

# Beam Search
print("Beam Search (num_beams=5):")
inputs = tokenizer(prompt, return_tensors='pt')
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        num_beams=5,           # 5개 후보 동시 탐색
        no_repeat_ngram_size=2,  # 2-gram 반복 방지
        pad_token_id=tokenizer.eos_token_id,
    )
result = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"  {result[:80]}")

print("\n-> Beam Search는 번역/요약에 적합, 창의적 생성에는 Sampling이 더 좋음")

프롬프트: 'The capital of France is'

Greedy (do_sample=False):
   the capital of the French Republic, and the capital of the French Republic is t

Beam Search (num_beams=5):
   Paris, which is home to some of the world's most prestigious universities, incl

-> Beam Search는 번역/요약에 적합, 창의적 생성에는 Sampling이 더 좋음


---
## 4. 실전 설정 가이드

In [15]:
# === 상황별 권장 설정 ===
configs = {
    "사실 기반 답변 (QA, 코드)": {
        "do_sample": False,  # Greedy
        "desc": "항상 일관된 답변, 창의성 불필요",
    },
    "일반 대화 (ChatBot)": {
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
        "desc": "적당한 다양성 + 일관성 균형",
    },
    "창의적 글쓰기": {
        "do_sample": True,
        "temperature": 1.0,
        "top_k": 50,
        "top_p": 0.95,
        "desc": "다양한 표현, 높은 창의성",
    },
    "번역/요약": {
        "num_beams": 4,
        "no_repeat_ngram_size": 3,
        "desc": "Beam Search로 정확도 높임",
    },
}

print("상황별 권장 generate() 설정:")
print("=" * 60)
for situation, config in configs.items():
    desc = config.pop('desc')
    print(f"\n  [{situation}]")
    print(f"    설명: {desc}")
    for k, v in config.items():
        print(f"    {k}: {v}")
    config['desc'] = desc  # 복원

상황별 권장 generate() 설정:

  [사실 기반 답변 (QA, 코드)]
    설명: 항상 일관된 답변, 창의성 불필요
    do_sample: False

  [일반 대화 (ChatBot)]
    설명: 적당한 다양성 + 일관성 균형
    do_sample: True
    temperature: 0.7
    top_p: 0.9

  [창의적 글쓰기]
    설명: 다양한 표현, 높은 창의성
    do_sample: True
    temperature: 1.0
    top_k: 50
    top_p: 0.95

  [번역/요약]
    설명: Beam Search로 정확도 높임
    num_beams: 4
    no_repeat_ngram_size: 3


In [16]:
# === logits 직접 확인: Phase 1에서 배운 softmax가 실제로 작동하는 모습 ===
prompt = "The meaning of"
inputs = tokenizer(prompt, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

# 마지막 토큰 위치의 logits (= 다음 토큰 확률 분포)
logits = outputs.logits[0, -1, :]  # [vocab_size]
probs = torch.softmax(logits, dim=-1)

# 상위 10개 후보 토큰
top_probs, top_ids = torch.topk(probs, 10)

print(f"프롬프트: '{prompt}'")
print(f"\n다음 토큰 후보 Top 10:")
for prob, idx in zip(top_probs, top_ids):
    token = tokenizer.decode([idx])
    print(f"  '{token}' (ID: {idx.item()}) : {prob.item()*100:.2f}%")

print(f"\n-> Phase 1에서 직접 softmax(logits / temperature)하던 것과 동일")
print(f"   generate()는 이 과정을 max_new_tokens회 자동 반복")

# 메모리 정리
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

프롬프트: 'The meaning of'

다음 토큰 후보 Top 10:
  ' the' (ID: 262) : 29.43%
  ' "' (ID: 366) : 11.04%
  ' this' (ID: 428) : 10.59%
  ' '' (ID: 705) : 4.07%
  ' these' (ID: 777) : 2.07%
  ' a' (ID: 257) : 2.04%
  ' that' (ID: 326) : 1.24%
  ' life' (ID: 1204) : 1.06%
  ' all' (ID: 477) : 0.62%
  ' his' (ID: 465) : 0.49%

-> Phase 1에서 직접 softmax(logits / temperature)하던 것과 동일
   generate()는 이 과정을 max_new_tokens회 자동 반복


---
## 정리

| 개념 | 핵심 |
|------|------|
| **정밀도** | FP16/BF16이 추론 기본, INT4는 VRAM 부족 시 |
| **device_map** | "auto"로 GPU/CPU 자동 분배 |
| **temperature** | 낮으면 확정적, 높으면 랜덤 |
| **top_p=0.9** | 실전 권장 설정 |
| **Beam Search** | 번역/요약용, 창의적 생성에는 Sampling |